In [11]:
from collections import defaultdict
corpus=[
    "best places to visit in india",
    "best places to visit in chennai",
    "best places to visit near me",
    "best places to visit during summer",
    "places to visit in kerala",
    "places to visit in goa",
    "best hotels to stay in chennai"
]
D=0.75
vocab=set()
for sentence in corpus:
    for word in sentence.lower().split():
        vocab.add(word)
vocab.add("<UNK>")
counts={}
for n in range(1,6):
    counts[n]=defaultdict(int)
history={}
for n in range(2,6):
    history[n]=defaultdict(int)
for sentence in corpus:
    words=sentence.lower().split()
    for i in range(len(words)):
        counts[1][(words[i],)]+=1
    for n in range(2,6):
        for i in range(len(words)-n+1):
            gram=tuple(words[i:i+n])
            hist=tuple(words[i:i+n-1])
            counts[n][gram]+=1
            history[n][hist]+=1
followers={}
continuation={}
for n in range(2,6):
    followers[n]=defaultdict(set)
    continuation[n]=defaultdict(set)
    for gram in counts[n]:
        hist=gram[:-1]
        word=gram[-1]
        followers[n][hist].add(word)
        continuation[n][word].add(hist)
def kneser_ney(history_words,word):
    n=len(history_words)+1
    if word not in vocab:
        word="<UNK>"
    if n==1:
        total=sum(counts[1].values())
        return counts[1].get((word,),0)/total
    hist=tuple(history_words)
    gram=hist+(word,)
    count=counts[n].get(gram,0)
    hist_count=history[n].get(hist,0)
    if hist_count==0:
        return kneser_ney(history_words[1:],word)
    first=max(count-D,0)/hist_count
    lam=(D*len(followers[n][hist]))/hist_count
    backoff=kneser_ney(history_words[1:],word)
    return first+lam*backoff
def autocomplete(text,k=5):
    words=text.lower().split()
    if len(words)>=4:
        context=tuple(words[-4:])
    else:
        context=tuple(words)
    scores=[]
    for word in vocab:
        if word=="<UNK>":
            continue
        score=kneser_ney(context,word)
        scores.append((score,word))
    scores.sort(reverse=True)
    print("\nSuggestions for:",text)
    for score,word in scores[:k]:
        print(word,"->",round(score,4))
autocomplete("best places to visit")
autocomplete("places to visit in")
autocomplete("best hotels to stay near")
    


Suggestions for: best places to visit
in -> 0.778
near -> 0.0988
during -> 0.0988
to -> 0.0052
visit -> 0.0044

Suggestions for: places to visit in
chennai -> 0.2627
kerala -> 0.172
india -> 0.172
goa -> 0.172
to -> 0.0443

Suggestions for: best hotels to stay near
me -> 0.2687
to -> 0.1312
visit -> 0.1125
places -> 0.1125
in -> 0.0938
